#### **OPTIMALIDAD DE LA SOLUCION - ALGORITMOS GREEDY**

In [23]:
import pandas as pd 
import matplotlib.pyplot as plt
from importlib import reload

import Clases.asignacion as asignacion_module
reload(asignacion_module)
from Clases.asignacion import Asignacion

import Clases.caja as caja_module
reload(caja_module)
from Clases.caja import Caja

import Clases.producto as producto_module
reload(producto_module)
from Clases.producto import Producto

import Clases.solucion as solucion_module
reload(solucion_module)
from Clases.solucion import Solucion

catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
especificaciones_cajas = pd.read_csv("Datos-finales/especificaciones_cajas.csv")
operaciones_planta = pd.read_csv("Datos-finales/operaciones_planta.csv")
procurement_cajas = pd.read_csv("Datos-finales/procurement_cajas.csv")
factibilidad = pd.read_csv("Factibilidad/factibilidad_4.5mm.csv")

Empecemos guardando los productos y tipos de cajas en listas en el estado actual, para cargarlos luego a las soluciones. Calculamos también las cajas asignables a cada producto en la lista de productos.

In [24]:
def guardar_cajas_y_productos(grosor):
    caja_compras_merge = especificaciones_cajas.merge(procurement_cajas,
                                                  on="caja_tipo_id")
    cajas = [
        Caja(
            caja_id = row["caja_tipo_id"],
            dim_interior_ancho = row["caja_interior_ancho"],
            dim_interior_largo = row["caja_interior_largo"],
            dim_interior_alto = row["caja_interior_alto"],
            costo_unitario = row['costo_unitario_base']
        )
        for _, row in caja_compras_merge.iterrows()
    ]

    prod_op_merge = catalogo_productos.merge(operaciones_planta, on="codigo_producto")
    productos = [
        Producto(
            codigo_producto = row['codigo_producto'],
            cantidad_paquetes = row['cantidad_paquetes'],
            peso_paquete = row['peso_neto_paquete'],
            demanda_buenos_aires = row['volumen_producto_planta_buenos_aires'],
            demanda_curitiba = row['volumen_producto_planta_curitiba'],
            demanda_santiago = row['volumen_producto_planta_santiago'],
            demanda_monterrey = row['volumen_producto_planta_monterrey'],
            demanda_bakersfield = row['volumen_producto_planta_bakersfield'],
            dim_producto_ancho = row['dim_producto_ancho'], 
            dim_producto_largo = row['dim_producto_largo'],
            dim_producto_alto = row['dim_producto_alto']
        )
        for _, row in prod_op_merge.iterrows()
    ]
    
    # Crear un diccionario de cajas por producto desde el CSV de factibilidad
    cajas_por_producto = {}
    for _, row in factibilidad.iterrows():
        codigo = row['codigo_producto']
        cajas_str = row['cajas_asignables_id']
        
        if isinstance(cajas_str, str) and cajas_str:
            # Separar por '; ' y limpiar
            cajas_list = [c.strip() for c in cajas_str.split(';') if c.strip()]
            cajas_por_producto[codigo] = cajas_list

    # Recorrer la lista de productos en orden y agregar las cajas
    for producto in productos:
        if producto.codigo_producto in cajas_por_producto:
            for caja_id in cajas_por_producto[producto.codigo_producto]:
                producto.agregar_caja_asignable(caja_id)
                
    # Elegir grosor
    for caja in cajas:
        caja.elegir_grosor(grosor_mm=grosor)
    return cajas, productos

#### **Greedy 0: Costos base sin descuentos**

Inicialmente, plantearemos una solución únicamente comparando los costos unitarios base, sin considerar todavía los descuentos por volúmenes anuales.

Como no hay ninguna restricción sobre la cantidad de cajas que podemos comprar de cada tipo, el problema se reduce en encontrar para cada producto el tipo de caja que más le convenga, es decir, el que ofrezca el menor costo.

Empecemos eligiendo un grosor de 5mm para todos los tipos de cajas, de manera que la restricción de headspace no supone ningún problema, pues las dimensiones internas de las cajas se diferencian hasta un 10% de las originales.

In [25]:
cajas, productos = guardar_cajas_y_productos(grosor=4.5)

Armamos una función general para buscar un tipo de caja por su id.

In [26]:
def buscar_caja_por_id(id):
    caja_a_buscar = None
    for caja in cajas:
        if caja.caja_id == id:
            caja_a_buscar = caja
    return caja_a_buscar

In [27]:
solucion_base = Solucion(grosor=4.5)

for producto in productos:
    for caja_id in producto.cajas_asignables:
        caja = buscar_caja_por_id(caja_id)
        asignacion = Asignacion(producto,caja)
        solucion_base.agregar_asignacion(asignacion, descuentos=False)

#solucion_base.exportar_submmit(nombre_csv="1-base_5mm")
solucion_base.resumen_por_asignacion()

,codigo_producto,demanda_total,cant_cajas_asignables,caja_id,utilizacion_pallet,utilizacion_caja,descuento_buenos_aires,descuento_curitiba,descuento_santiago,descuento_monterrey,descuento_bakersfield,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,6,02cf77de65b70dd77905e2e33d78478f,0.385062,0.983137,0.0,0.0,0.0,0.0,0.0,927967.80,85924,12888600,13816567.80
1,BR0001,1546613,6,5e55b0f0e0122b55ba8d4d91fb921c5a,0.679340,0.927148,0.0,0.0,0.0,0.0,0.0,1005298.45,51555,7733250,8738548.45
2,BR0001,1546613,6,60eb77c934f99a121f410ec2037a2a46,0.336930,0.934944,0.0,0.0,0.0,0.0,0.0,927967.80,103109,15466350,16394317.80
3,BR0001,1546613,6,8f26468860664ecfed551104179d7f3e,0.882030,0.952830,0.0,0.0,0.0,0.0,0.0,1005298.45,38667,5800050,6805348.45
4,BR0001,1546613,6,a23eeef1c0cd8cfd40b780d65382034a,0.841569,1.000000,0.0,0.0,0.0,0.0,0.0,1005298.45,38667,5800050,6805348.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5114,BR0421,145803,7,371bc8e13ebaec61128f288334956fa5,0.817911,0.978895,0.0,0.0,0.0,0.0,0.0,94771.95,2604,390600,485371.95
5115,BR0421,145803,7,3826faf22304dbe5bef22f8c36155874,0.400682,1.000000,0.0,0.0,0.0,0.0,0.0,87481.80,5208,781200,868681.80
5116,BR0421,145803,7,9dbb448086386afae3321dda0441c688,0.411000,0.973913,0.0,0.0,0.0,0.0,0.0,87481.80,5208,781200,868681.80
5117,BR0421,145803,7,a7d151b83674a2f9c36356597a93821c,0.802831,0.997536,0.0,0.0,0.0,0.0,0.0,102062.10,2604,390600,492662.10


#### **Greedy 1: Elección por menor costo**

In [28]:
def solucion_greedy(grosor, cajas, productos_ordenados, criterio):

    solucion = Solucion(grosor=grosor)

    for producto in productos_ordenados:
        metricas = {}  # caja_id -> (volumen_total, costo_packaging, costo_flete, costo_total, utilizacion_pallet)
        
        for caja_id in producto.cajas_asignables:
            caja = buscar_caja_por_id(caja_id)
            
            # Volumen requerido actual (antes de asignar este producto)
            volumen_total = caja.unidades_total_requeridas()
            utilizacion_pallet = caja.utilizacion_pallet()
            
            # Costo simulado de asignar este producto a esta caja
            asignacion = Asignacion(producto, caja)
            caja.asignar_producto(producto)
            
            # Calculamos los costos del producto utilizando ese tipo de caja
            costo_packaging = asignacion.costo_packaging_producto_total()
            costo_flete = 150 * asignacion.cant_pallets_requeridas()
            costo_total = costo_packaging + costo_flete
            metricas[caja_id] = (volumen_total, costo_packaging, costo_flete, costo_total, utilizacion_pallet)
            
            # Revertimos la asignación para que no se aplique un descuento posterior
            caja.revocar_producto(producto)
        
        if criterio == "minimizar_costo_total":
            caja_id_optima = min(metricas, key=lambda id: (metricas[id][3]))
        elif criterio == "maximizar_volumen_tipo":        
            caja_id_optima = max(metricas, key=lambda id: (metricas[id][0], -metricas[id][3]))
        elif criterio == "minimizar_costo_flete":
            caja_id_optima = min(metricas, key=lambda id: (metricas[id][2]))
        elif criterio == "maximizar_utilizacion_pallet":
            caja_id_optima = max(metricas, key=lambda id: (metricas[id][4]))
        
        caja_optima = buscar_caja_por_id(caja_id_optima)
        
        asignacion = Asignacion(producto, caja_optima)
        solucion.agregar_asignacion(asignacion)    
    
    return solucion

Guardamos nuevamente cajas y productos, reiniciando toda asignación previa:

In [29]:
cajas, productos = guardar_cajas_y_productos(grosor=4.5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: len(p.cajas_asignables))

solucion_greedy1 = solucion_greedy(4.5, cajas, productos_ordenados, criterio="minimizar_costo_total")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy1.resumen_por_asignacion()
solucion_greedy1.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 4.5mm
Número de tipos de cajas distintos: 53
Costo packaging: 29601910.84500001
Costo flete: 185904450
Costo total: 215506360.845
Utilización de pallet promedio: 0.8545926228450929
Utilización de caja promedio: 0.9740985799432961


In [30]:
cajas, productos = guardar_cajas_y_productos(grosor=4.5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: len(p.cajas_asignables))

solucion_greedy2 = solucion_greedy(4.5, cajas, productos_ordenados, criterio="minimizar_costo_flete")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy2.resumen_por_asignacion()
solucion_greedy2.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 4.5mm
Número de tipos de cajas distintos: 62
Costo packaging: 29978691.959999982
Costo flete: 185904450
Costo total: 215883141.95999998
Utilización de pallet promedio: 0.8516818614193458
Utilización de caja promedio: 0.9773307749235433


In [31]:
cajas, productos = guardar_cajas_y_productos(grosor=4.5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: len(p.cajas_asignables))

solucion_greedy3 = solucion_greedy(4.5, cajas, productos_ordenados, criterio="maximizar_utilizacion_pallet")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy3.resumen_por_asignacion()
solucion_greedy3.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 4.5mm
Número de tipos de cajas distintos: 48
Costo packaging: 29467682.324999996
Costo flete: 185909100
Costo total: 215376782.325
Utilización de pallet promedio: 0.8655753879652118
Utilización de caja promedio: 0.9611779188020872


In [32]:
solucion_greedy3.resumen_por_asignacion()

,codigo_producto,demanda_total,cant_cajas_asignables,caja_id,utilizacion_pallet,utilizacion_caja,descuento_buenos_aires,descuento_curitiba,descuento_santiago,descuento_monterrey,descuento_bakersfield,costo_packaging,cant_pallets,costo_flete,costo_total
0,BR0001,1546613,6,8f26468860664ecfed551104179d7f3e,0.882030,0.952830,-0.3,-0.3,-0.2,-0.1,0.1,716913.730,38667,5800050,6516963.730
1,BR0002,139211,7,3f4b03b56bb388655f2cb1495b69f5d7,0.939997,0.978573,-0.3,-0.2,-0.3,0.1,-0.1,72389.720,2901,435150,507539.720
2,BR0003,172506,17,67894c5a8355ca9b7e57b96362fa45f1,0.971657,0.970414,-0.2,-0.3,-0.2,0.0,-0.2,89703.120,2157,323550,413253.120
3,BR0004,271715,5,72188210fb21e6ddf6d4aba9a9301c48,0.947639,0.980652,0.1,-0.3,0.1,-0.1,-0.3,123630.325,4853,727950,851580.325
4,BR0005,7586,23,e205014ff79e36431f829f1fab46061c,0.923620,0.915560,0.1,-0.2,-0.3,0.1,0.1,3944.720,159,23850,27794.720
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
422,BR0417,2761,18,35174969e008414733b7133a380c2f8c,0.852133,0.928834,-0.3,-0.3,-0.3,-0.3,-0.3,1256.255,50,7500,8756.255
423,BR0418,3942,18,35174969e008414733b7133a380c2f8c,0.852133,0.928834,-0.3,-0.3,-0.3,-0.3,-0.3,1793.610,71,10650,12443.610
424,BR0419,43300,6,a12aea41ad9864b83129274495d5d5b2,0.948185,0.983768,0.1,-0.1,0.1,-0.1,0.1,25330.500,602,90300,115630.500
425,BR0420,205852,6,8b92b5a6728066959f3f11c35d1cd62f,0.746896,0.979315,-0.1,-0.2,0.1,-0.3,0.1,102563.370,3677,551550,654113.370


In [33]:
solucion_greedy3.exportar_submmit(nombre_csv="1-greedy_4.5mm_max_utilizacion_pallet")

In [34]:
cajas, productos = guardar_cajas_y_productos(grosor=4.5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: len(p.cajas_asignables))

solucion_greedy4 = solucion_greedy(4.5, cajas, productos_ordenados, criterio="maximizar_volumen_tipo")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy4.resumen_por_asignacion()
solucion_greedy4.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 4.5mm
Número de tipos de cajas distintos: 42
Costo packaging: 29352770.840000026
Costo flete: 207770550
Costo total: 237123320.84000003
Utilización de pallet promedio: 0.8103098062243965
Utilización de caja promedio: 0.9641227743503185


In [35]:
cajas, productos = guardar_cajas_y_productos(grosor=4.5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)

solucion_greedy5 = solucion_greedy(4.5, cajas, productos_ordenados, criterio="minimizar_costo_total")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy5.resumen_por_asignacion()
solucion_greedy5.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 4.5mm
Número de tipos de cajas distintos: 53
Costo packaging: 29688235.739999972
Costo flete: 185904450
Costo total: 215592685.73999998
Utilización de pallet promedio: 0.8541194160870823
Utilización de caja promedio: 0.9746951185048611


In [36]:
cajas, productos = guardar_cajas_y_productos(grosor=4.5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)

solucion_greedy6 = solucion_greedy(4.5, cajas, productos_ordenados, criterio="minimizar_costo_flete")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy6.resumen_por_asignacion()
solucion_greedy6.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 4.5mm
Número de tipos de cajas distintos: 62
Costo packaging: 29978691.95999997
Costo flete: 185904450
Costo total: 215883141.95999998
Utilización de pallet promedio: 0.8516818614193459
Utilización de caja promedio: 0.9773307749235436


In [37]:
cajas, productos = guardar_cajas_y_productos(grosor=4.5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)

solucion_greedy7 = solucion_greedy(4.5, cajas, productos_ordenados, criterio="maximizar_utilizacion_pallet")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy7.resumen_por_asignacion()
solucion_greedy7.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 4.5mm
Número de tipos de cajas distintos: 48
Costo packaging: 29467682.324999988
Costo flete: 185909100
Costo total: 215376782.325
Utilización de pallet promedio: 0.8655753879652116
Utilización de caja promedio: 0.961177918802087


In [38]:
cajas, productos = guardar_cajas_y_productos(grosor=4.5)
# Ordenamos los productos según la cantidad de cajas asignables de menor a mayor
productos_ordenados = sorted(productos, key=lambda p: p.demanda_total(), reverse=True)

solucion_greedy8 = solucion_greedy(4.5, cajas, productos_ordenados, criterio="maximizar_volumen_tipo")

#solucion_greedy1.exportar_submmit(nombre_csv="2-greedy1_5mm")
solucion_greedy8.resumen_por_asignacion()
solucion_greedy8.resumen_general()

Situación original
--------------------------------------------------
Número de tipos de cajas distintos: 204
Costo packaging: 30295472.424999997
Costo flete: 179068800
Costo total: 209364272.425
Utilización de pallet promedio: 0.8320794116391749
Utilización de caja promedio: 1.0

Situación nueva
--------------------------------------------------
Grosor elegido: 4.5mm
Número de tipos de cajas distintos: 50
Costo packaging: 29652289.24499998
Costo flete: 186964950
Costo total: 216617239.24499997
Utilización de pallet promedio: 0.85408743641051
Utilización de caja promedio: 0.9709951459072738
